# 05 — Entraînement baseline vision (KneeNet)

Lance `src/train/train.py` avec `configs/baseline.yaml` et affiche les courbes.

- Entrée : 3 plans (sag/cor/ax, fluid prioritaire) × 24 slices × 224², normalisés GPU
- Modèle : ConvNeXt-tiny partagé (24 canaux = slices) → GAP → concat 3×768 → MLP → 12 logits
- Loss : BCE masquée (-1 = ignoré) + pos_weight + label smoothing
- Métrique : macro-AUC sur hold-out pseudo-labels ; sanity check final sur les 58 gold

Sur RTX 3070 (8 Go) : batch 16 ≈ 4 Go VRAM, ~3–4 min/epoch.

In [ ]:
# L'entrainement se lance en terminal (plus robuste que dans le notebook) :
#   python -X utf8 -m src.train.train --config configs/baseline.yaml
# Cette cellule attend la fin et affiche l'historique.
import json, time
from pathlib import Path

run_dir = Path.cwd().parent / 'checkpoints' / 'baseline_convnext_tiny'
hist_path = run_dir / 'history.json'
print('en attente de', hist_path, '...')
while not hist_path.exists():
    time.sleep(30)
print('termine')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

hist = pd.DataFrame(json.loads(hist_path.read_text()))
display(hist.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist.epoch, hist.loss, marker='o'); axes[0].set_title('loss train')
axes[1].plot(hist.epoch, hist.val_macro_auc, marker='o', color='green')
axes[1].set_title('val macro-AUC (hold-out pseudo-labels)')
plt.tight_layout()

In [ ]:
# Sanity check gold (58 etudes, jamais vues, jamais utilisees pour piloter)
gold_path = run_dir / 'gold_sanity.json'
if gold_path.exists():
    gold = json.loads(gold_path.read_text())
    print('macro-AUC gold :', round(gold['macro_auc'], 4))
    display(pd.Series(gold['per_target']).sort_values(ascending=False).round(3))